# Task 2, Step 5: Stage 3, Natural VQA Specialization

The final 750 updates continue the same connector and adapters. Each draw: natural VQAv2 70%, presence, counting, and caption replay 10% each. Connector peak LR $5\times10^{-5}$, LoRA peak LR $10^{-5}$.

The completed Stage 3 model is the final model. Checkpoints are never selected using final-test results.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task2_vlm_grounding")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task2_vlm_grounding.src.notebook_utils import *
setup_notebook()

In [ ]:
from task2_vlm_grounding.src.notebook_utils import load_run
run = load_run()
history, dev, summary = run["history"], run["dev"], run["summary"]
scope = history[history["stage"] == 3]
if scope.empty:
    raise RuntimeError("Stage 3 has not been trained yet.")
print("Updates logged:", len(history), "of", summary["total_updates"], "| full recipe:", summary["full_recipe"])
print("Trainable parameters by stage:", summary["stage_params"], "| peak GPU MB:", round(summary["peak_gpu_mb"], 1))
print("Skipped updates:", len(summary["skipped_updates"]))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
axes[0].plot(scope["update"], scope["loss"], alpha=0.25, color="#3498db")
axes[0].plot(scope["update"], scope["loss"].rolling(25, min_periods=1).mean(), color="#2c3e50", label="25-update mean")
axes[0].set(title="Training loss (example-weighted)", xlabel="Main-run update", ylabel="Cross-entropy"); axes[0].legend()
axes[1].plot(scope["update"], scope["connector_lr"], label="Connector")
if scope["lora_lr"].notna().any():
    axes[1].plot(scope["update"], scope["lora_lr"], label="LoRA")
axes[1].set(title="Learning rates: 5% warmup, cosine to 10% of peak", xlabel="Main-run update", ylabel="LR", yscale="log"); axes[1].legend()
axes[2].plot(scope["update"], scope["grad_norm"], color="#e67e22", alpha=0.5)
axes[2].axhline(1.0, color="crimson", linestyle="--", label="Clip at 1.0")
axes[2].set(title="Gradient norm before clipping", xlabel="Main-run update", ylabel="Norm", yscale="log"); axes[2].legend()
save_figure("task2_stage3_training")

## 5.1 The whole curriculum
Stage boundaries are shaded. Look for what carries over between stages and what changes.

In [ ]:
boundaries = np.cumsum([summary["total_updates"] * s for s in [0.5, 1 / 3, 1 / 6]]).round().astype(int).tolist()
fig, axes = plt.subplots(2, 2, figsize=(16, 8.2), sharex=True)
axes[0, 0].plot(history["update"], history["loss"].rolling(25, min_periods=1).mean(), color="#2c3e50")
axes[0, 0].set(title="Training loss (25-update mean)", ylabel="Cross-entropy")
axes[0, 1].plot(dev["update"], dev["caption_loss"], marker="o", color="#2980b9")
axes[0, 1].set(title="Development caption loss", ylabel="Cross-entropy")
for suite, marker in [("grounding", "o"), ("natural", "s")]:
    for condition, color in [("correct", "#2ecc71"), ("swap", "#e74c3c"), ("no_image", "#95a5a6")]:
        axes[1, 0].plot(dev["update"], dev[f"{suite}_nll_{condition}"], marker=marker, color=color, linestyle="-" if suite == "grounding" else "--", label=f"{suite}: {condition}")
    axes[1, 1].plot(dev["update"], dev[f"{suite}_swap_penalty"], marker=marker, label=suite)
axes[1, 0].set(title="Development reference-answer NLL", xlabel="Main-run update", ylabel="NLL"); axes[1, 0].legend(fontsize=8, ncol=2)
axes[1, 1].axhline(0, color="black")
axes[1, 1].set(title="Image-swap penalty", xlabel="Main-run update", ylabel=r"$\Delta$NLL"); axes[1, 1].legend()
for ax in axes.flat:
    shade_stages(ax, boundaries)
save_figure("task2_curriculum_learning_curves")
dev.round(4).to_csv(f"{RESULTS_DIR}/development_metrics.csv", index=False)

## 5.2 Resources

In [ ]:
resources = pd.DataFrame([{"Stage": k, "Trainable parameters": v, "Cumulative training min": summary["stage_seconds"].get(k, float("nan")) / 60, "Saved weights MB": summary["checkpoint_mb"].get(f"stage_{k}")} for k, v in summary["stage_params"].items()])
resources["Peak GPU MB (whole run)"] = summary["peak_gpu_mb"]
resources.to_csv(f"{RESULTS_DIR}/training_resources.csv", index=False)
display(resources.round(2))